In [3]:
import pandas as pd 
import numpy as np

In [1]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


In [5]:
df = pd.read_csv("hour.csv")

In [6]:
df.head()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


In [ ]:
data.tail()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
17374,17375,2012-12-31,1,1,12,19,0,1,1,2,0.26,0.2576,0.60,0.1642,11,108,119
17375,17376,2012-12-31,1,1,12,20,0,1,1,2,0.26,0.2576,0.60,0.1642,8,81,89
17376,17377,2012-12-31,1,1,12,21,0,1,1,1,0.26,0.2576,0.60,0.1642,7,83,90
17377,17378,2012-12-31,1,1,12,22,0,1,1,1,0.26,0.2727,0.56,0.1343,13,48,61
17378,17379,2012-12-31,1,1,12,23,0,1,1,1,0.26,0.2727,0.65,0.1343,12,37,49


In [9]:
data = data.sort_values(["dteday","hr"]).reset_index(drop= True)

In [10]:
features = [
    "season",
    "yr",
    "mnth",
    "hr",
    "holiday",
    "weekday",
    "workingday",
    "weathersit",
    "temp",
    "atemp",
    "hum",
    "windspeed",
]

In [11]:
X = data[features]
y = data['cnt']

In [12]:
split_index = int(len(data)*0.8)

In [13]:
X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

In [14]:
categorical_features = [
    "season",
    "yr",
    "mnth",
    "hr",
    "holiday",
    "weekday",
    "workingday",
    "weathersit",
]

In [15]:
numeric_features = ["temp", "atemp", "hum", "windspeed"]

In [16]:
preprocessor = ColumnTransformer(
    transformers=[
        ("categories", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numbers", "passthrough", numeric_features),
    ]
)

In [17]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=100,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

In [18]:
model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('categories', ...), ('numbers', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [19]:
predictions = model.predict(X_test)

In [20]:
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test,predictions))
r2 = r2_score(y_test, predictions)

In [21]:
print(f"Dataset rows: {len(data)}")


Dataset rows: 17379


In [22]:
print(f"Training rows: {len(X_train)}")

Training rows: 13903


In [23]:
print(f"Testing rows: {len(X_test)}")

Testing rows: 3476


In [24]:
print(f"MAE:  {mae:.2f} bikes")

MAE:  55.01 bikes


In [25]:
print(f"RMSE: {rmse:.2f} bikes")


RMSE: 82.90 bikes


In [26]:
print(f"R²:   {r2:.3f}")

R²:   0.859


In [27]:
results = pd.DataFrame(
    {
        "Actual rentals": y_test.iloc[:10].values,
        "Predicted rentals": np.round(predictions[:10]).astype(int),
    }
)

In [28]:
print("\nFirst 10 test predictions:")
print(results.to_string(index=False))


First 10 test predictions:
 Actual rentals  Predicted rentals
            283                271
            253                273
            261                281
            306                272
            445                329
            868                752
            814                719
            610                529
            448                422
            317                284
